In [10]:
import pandas as pd

file_path = "/Users/apple/Desktop/Transcribes/group_15_url_1_transcript.csv"
df = pd.read_csv(file_path)

df = df[df["Sentence"] != "PYM JBZ"].reset_index(drop=True)

print(df.head(10))

                   Start Time                    End Time  \
0  1900-01-01 00:09:18.657982  1900-01-01 00:09:20.657982   
1  1900-01-01 00:09:20.657982  1900-01-01 00:09:22.657982   
2  1900-01-01 00:09:22.657982  1900-01-01 00:09:24.657982   
3  1900-01-01 00:09:24.657982  1900-01-01 00:09:26.657982   
4  1900-01-01 00:09:26.657982  1900-01-01 00:09:28.657982   
5  1900-01-01 00:09:28.657982  1900-01-01 00:09:30.657982   
6  1900-01-01 00:09:30.657982  1900-01-01 00:09:32.657982   
7  1900-01-01 00:09:32.657982  1900-01-01 00:09:34.657982   
8  1900-01-01 00:09:34.657982  1900-01-01 00:09:36.657982   
9  1900-01-01 00:09:36.657982  1900-01-01 00:09:38.657982   

                      Sentence                                Translation  \
0     مثلاً خیلی ما دوست داریم            For example, we really like it.   
1        خیلی هم دوستشون داریم                    We really love them too   
2       خیلی هم کنار همدیگه هم     Very much together with each other too   
3      خیلی به همون 

### Part of speech tagging (pos tagging - persian)

In [2]:
import stanza

stanza.download('fa')

/opt/anaconda3/envs/nlp_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-19 14:41:06 INFO: Downloaded file to /Users/apple/stanza_resources/resources.json
2025-09-19 14:41:06 INFO: Downloading default packages for language: fa (Persian) ...
2025-09-19 14:41:07 INFO: File exists: /Users/apple/stanza_resources/fa/default.zip
2025-09-19 14:41:09 INFO: Finished downloading models and saved to /Users/apple/stanza_resources


In [3]:
nlp = stanza.Pipeline('fa')

doc = nlp("من عاشق یادگیری ماشین هستم")

for sent in doc.sentences:
    for word in sent.words:
        print(word.text, word.upos)

2025-09-19 14:41:11 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-09-19 14:41:11 INFO: Downloaded file to /Users/apple/stanza_resources/resources.json
2025-09-19 14:41:12 INFO: Loading these models for language: fa (Persian):
| Processor | Package        |
------------------------------
| tokenize  | perdt          |
| mwt       | perdt          |
| pos       | perdt_charlm   |
| lemma     | perdt_nocharlm |
| depparse  | perdt_charlm   |
| ner       | arman          |

2025-09-19 14:41:12 INFO: Using device: cpu
2025-09-19 14:41:12 INFO: Loading: tokenize
2025-09-19 14:41:13 INFO: Loading: mwt
2025-09-19 14:41:13 INFO: Loading: pos
2025-09-19 14:41:16 INFO: Loading: lemma
2025-09-19 14:41:17 INFO: Loading: depparse
2025-09-19 14:41:17 INFO: Loading: ner
2025-09-19 14:41:20 INFO: Done loading processors!


من PRON
عاشق ADJ
یادگیری NOUN
ماشین NOUN
هستم VERB


In [13]:
nlp = stanza.Pipeline('fa', processors='tokenize,pos', verbose=False)

def get_pos_tags(text):
    doc = nlp(str(text))
    tags = []
    for sentence in doc.sentences:
        for word in sentence.words:
            tags.append(f"{word.text}/{word.upos}")
    return " ".join(tags)

df["POS_Tags"] = df["Sentence"].apply(get_pos_tags)

from IPython.display import display
display(df[["Sentence", "POS_Tags"]].head(10))

,Sentence,POS_Tags
0,مثلاً خیلی ما دوست داریم,مثلاً/ADV خیلی/ADV ما/PRON دوست/NOUN داریم/VERB
1,خیلی هم دوستشون داریم,خیلی/ADV هم/ADV دوستشون/NOUN داریم/VERB
2,خیلی هم کنار همدیگه هم,خیلی/ADV هم/ADV کنار/NOUN همدیگه/PRON هم/ADV
3,خیلی به همون خوش میگذره,خیلی/ADV به/ADP همون/PRON خوش/NOUN میگذره/VERB
4,حالا چطور شغل من,حالا/NOUN چطور/ADV شغل/NOUN من/PRON
5,چطور فضای اجتماعی من,چطور/ADV فضای/NOUN اجتماعی/ADJ من/PRON
6,منظور عرض من این بود,منظور/NOUN عرض/NOUN من/PRON این/PRON بود/AUX
7,که اون آدمهایی رو که,که/SCONJ اون/DET آدمهایی/NOUN رو/ADP که/SCONJ
8,میتونیم روشون حساب کنیم,میتونیم/VERB روشون/ADV حساب/NOUN کنیم/VERB
9,شاید اون دوست هم کسایی باشن,شاید/ADV اون/DET دوست/NOUN هم/ADV کسایی/NOUN ب...


### TF-IDF - Persian

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

sentences = df["Sentence"].astype(str).tolist()

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)

feature_names = vectorizer.get_feature_names_out()

tfidf_dicts = []
for row in tfidf_matrix.toarray():
    tfidf_dict = {feature_names[i]: row[i] for i in range(len(feature_names)) if row[i] > 0}
    tfidf_dicts.append(tfidf_dict)

df["TF_IDF"] = tfidf_dicts

display(df[["Sentence", "TF_IDF"]].head(10))

,Sentence,TF_IDF
0,مثلاً خیلی ما دوست داریم,"{'خیلی': 0.34521385765844076, 'داریم': 0.49293..."
1,خیلی هم دوستشون داریم,"{'خیلی': 0.3665005429428216, 'داریم': 0.523327..."
2,خیلی هم کنار همدیگه هم,"{'خیلی': 0.29590832228130076, 'هم': 0.56564646..."
3,خیلی به همون خوش میگذره,"{'به': 0.2807735067980976, 'خوش': 0.4757911821..."
4,حالا چطور شغل من,"{'حالا': 0.41792869038902736, 'شغل': 0.6098554..."
5,چطور فضای اجتماعی من,"{'اجتماعی': 0.5849084104784257, 'فضای': 0.5324..."
6,منظور عرض من این بود,"{'این': 0.29828397267569684, 'بود': 0.36277415..."
7,که اون آدمهایی رو که,"{'آدمهایی': 0.6658645560036472, 'اون': 0.38830..."
8,میتونیم روشون حساب کنیم,"{'حساب': 0.4900865483411329, 'روشون': 0.526871..."
9,شاید اون دوست هم کسایی باشن,"{'اون': 0.31836755648728077, 'باشن': 0.4811576..."


### Sentiment Analysis on Persian Text with ParsBERT

```python
# Step 1: Install required libraries (in Colab)
!pip install torch --upgrade
!pip install transformers
# Step 2: Import libraries
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
# Step 3: Load dataset in Colab
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("group_15_url_1_transcript.csv")
# Step 4: Load ParsBERT Sentiment Model
model_name = "HooshvareLab/bert-fa-base-uncased-sentiment-snappfood"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_model = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
# Step 5: Apply Sentiment Analysis
df["Sentiment"] = df["Sentence"].astype(str).apply(lambda x: sentiment_model(x)[0]["label"])
# Step 6: Save results
df.to_csv("sentiment_results.csv", index=False)
files.download("sentiment_results.csv")


Since my local environment (VS Code on MacBook) had compatibility issues with PyTorch and ParsBERT, I could not run the model locally.
To solve this, I moved the workflow to Google Colab, where I installed the required libraries and ran the model without errors.
Here is what I did:
Uploaded my original dataset (group_15_url_1_transcript.csv) into Colab.
Loaded the ParsBERT sentiment model (HooshvareLab/bert-fa-base-uncased-sentiment-snappfood).
Applied sentiment analysis to the Sentence column and got sentiment labels (positive, negative, or neutral).
Saved the results into a new CSV file (sentiment_results.csv) and downloaded it back to my computer.
Opened VS Code, loaded both the original dataset and the new sentiment results, and merged them into one dataframe (df_final).
Saved this as final_dataset.csv, which now contains all original columns plus the new Sentiment column.

### Pretrained Embeddings 

In [22]:
from gensim.models import KeyedVectors

file_path = "/Users/apple/Desktop/Transcribes/cc.fa.300.vec"

word_vectors = KeyedVectors.load_word2vec_format(file_path, binary=False, limit=50000)

print(word_vectors['ایران'])
print(word_vectors.most_similar("دانشگاه"))


[-0.0068  0.0296 -0.0508 -0.0148  0.018   0.1155  0.0347 -0.0229  0.0228
  0.1124  0.0125  0.0105 -0.0414 -0.0095  0.0032 -0.0425  0.0355  0.0185
  0.006   0.0164 -0.0045  0.0159  0.0351  0.056  -0.023   0.0123 -0.124
 -0.0278  0.0112 -0.0161 -0.0264 -0.0282 -0.0295 -0.017   0.0097  0.0151
  0.053   0.0193 -0.016   0.045  -0.0077 -0.0136  0.006  -0.0065  0.0509
 -0.0142  0.013  -0.0192 -0.0154 -0.0017 -0.0372  0.0409 -0.0031  0.0052
 -0.0603 -0.0005 -0.0604 -0.01    0.028  -0.015   0.0113 -0.0151 -0.0124
 -0.0086 -0.0153  0.011   0.04    0.0244  0.0541 -0.0155  0.0133 -0.0068
  0.0907 -0.0234 -0.0344 -0.0244  0.0246  0.0226  0.0305 -0.0393  0.0104
 -0.002  -0.0145  0.0815  0.0061 -0.0272 -0.03   -0.0251 -0.0048 -0.0266
 -0.0123 -0.0096 -0.0325 -0.0165  0.0643  0.0053  0.0582 -0.0084 -0.0022
 -0.0018 -0.0346  0.0125 -0.0246  0.      0.0659  0.0009  0.0014 -0.0074
 -0.0061 -0.035  -0.0613  0.0161  0.0221  0.0409  0.0087 -0.0603  0.004
  0.0029 -0.0038 -0.0002 -0.0079 -0.0472 -0.0064 -0.0

In [23]:
word_vectors.save("cc_fa_300.kv")
word_vectors = KeyedVectors.load("cc_fa_300.kv")


In [27]:
import numpy as np

def get_sentence_embedding(sentence, model):
    words = str(sentence).split()
    vectors = []
    for word in words:
        if word in model:
            vectors.append(model[word])
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)  
    else:
        return np.zeros(model.vector_size)

df["Embedding"] = df["Sentence"].apply(lambda x: get_sentence_embedding(x, word_vectors))

print(df[["Sentence", "Embedding"]].head(10))



                      Sentence  \
0     مثلاً خیلی ما دوست داریم   
1        خیلی هم دوستشون داریم   
2       خیلی هم کنار همدیگه هم   
3      خیلی به همون خوش میگذره   
4             حالا چطور شغل من   
5         چطور فضای اجتماعی من   
6         منظور عرض من این بود   
7         که اون آدمهایی رو که   
8      میتونیم روشون حساب کنیم   
9  شاید اون دوست هم کسایی باشن   

                                           Embedding  
0  [-0.057, -0.026459998, -0.0165, 0.0012800004, ...  
1  [-0.061366666, 0.047933336, -0.00073333207, 0....  
2  [-0.05068, 0.044420004, 0.016940001, 0.05422, ...  
3  [0.0043200008, 0.1136, -0.018819999, -0.01594,...  
4  [-0.019724999, -0.005324997, -0.0586, -0.03890...  
5  [-0.032849997, -0.023399998, -0.059550002, -0....  
6  [0.028220002, 0.033500005, -0.0626, -0.04044, ...  
7  [-0.108820006, -0.08222, 0.027359998, -0.00626...  
8  [0.010375001, 0.13582501, -0.01075, 0.03055, -...  
9  [-0.010133332, 0.07028333, 0.01865, 0.03613333...  


### Custome word embedding model

In [29]:
from gensim.models import Word2Vec

sentences_tokenized = [str(s).split() for s in df["Sentence"].astype(str).tolist()]

print(sentences_tokenized[:3])  


[['مثلاً', 'خیلی', 'ما', 'دوست', 'داریم'], ['خیلی', 'هم', 'دوستشون', 'داریم'], ['خیلی', 'هم', 'کنار', 'همدیگه', 'هم']]


In [30]:
# Word2Vec 
custom_w2v = Word2Vec(
    sentences=sentences_tokenized, 
    vector_size=100,  
    window=5,          
    min_count=1,       
    workers=4,         
    sg=1               
)

print(custom_w2v.wv["من"])


[-2.15474628e-02  4.15059701e-02 -7.20854849e-03  2.00440381e-02
  1.04057575e-02 -6.37107566e-02  2.52453368e-02  1.01070687e-01
 -3.71722318e-02 -3.45337205e-02 -1.40638389e-02 -5.83336912e-02
  8.29987880e-03  1.50558772e-02  3.10973208e-02 -1.54036656e-02
  3.94443236e-02 -1.78382751e-02 -1.07452841e-02 -7.03180507e-02
  7.89603591e-03 -1.81495177e-03  3.28424536e-02 -5.18126749e-02
 -2.33193357e-02  1.19386883e-02 -4.81893159e-02 -1.39936637e-02
 -3.84750664e-02  1.13510946e-02  4.79026698e-02 -1.70348063e-02
  1.21485973e-02 -4.03972156e-02 -4.40702960e-03  5.18232547e-02
 -3.36840679e-03 -2.04938408e-02 -1.29850814e-03 -4.89806719e-02
  2.42192131e-02 -3.90345268e-02 -3.67145166e-02  1.11071831e-02
  2.53469124e-02 -1.35592464e-03 -1.79632772e-02 -6.05797954e-03
  3.03944461e-02  1.47990044e-02  2.63751578e-02 -3.78090627e-02
 -3.04669637e-04 -6.46729209e-03 -4.06738138e-03  2.81900745e-02
  2.08302196e-02 -6.71831518e-03 -2.76270267e-02  1.42159276e-02
 -5.30405436e-03  1.09965

In [31]:
import numpy as np

def get_custom_sentence_embedding(sentence, model):
    words = str(sentence).split()
    vectors = []
    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

df["Custom_Embedding"] = df["Sentence"].apply(lambda x: get_custom_sentence_embedding(x, custom_w2v))

print(df[["Sentence", "Custom_Embedding"]].head(10))


                      Sentence  \
0     مثلاً خیلی ما دوست داریم   
1        خیلی هم دوستشون داریم   
2       خیلی هم کنار همدیگه هم   
3      خیلی به همون خوش میگذره   
4             حالا چطور شغل من   
5         چطور فضای اجتماعی من   
6         منظور عرض من این بود   
7         که اون آدمهایی رو که   
8      میتونیم روشون حساب کنیم   
9  شاید اون دوست هم کسایی باشن   

                                    Custom_Embedding  
0  [-0.0029616677, 0.020769496, -0.0063136173, 0....  
1  [-0.0029392797, 0.021788679, -0.008537899, 0.0...  
2  [-0.009498013, 0.021373184, -0.009999054, 0.00...  
3  [-0.008988088, 0.0215974, -0.0065619564, 0.008...  
4  [-0.0040985118, 0.01274536, -0.0028300341, 0.0...  
5  [-0.00723907, 0.011048229, -0.002842893, 0.002...  
6  [-0.007640221, 0.024557415, -0.0040454664, 0.0...  
7  [-0.010853429, 0.036431305, -0.0070511675, 0.0...  
8  [-0.0040984196, 0.0026261974, 0.0001800225, -0...  
9  [-0.008134628, 0.017070876, -0.0042339857, 0.0...  


In [33]:
print(df)

                      Start Time                    End Time  \
0     1900-01-01 00:09:18.657982  1900-01-01 00:09:20.657982   
1     1900-01-01 00:09:20.657982  1900-01-01 00:09:22.657982   
2     1900-01-01 00:09:22.657982  1900-01-01 00:09:24.657982   
3     1900-01-01 00:09:24.657982  1900-01-01 00:09:26.657982   
4     1900-01-01 00:09:26.657982  1900-01-01 00:09:28.657982   
...                          ...                         ...   
1655  1900-01-01 01:14:08.605986  1900-01-01 01:14:10.605986   
1656  1900-01-01 01:14:10.605986  1900-01-01 01:14:12.605986   
1657  1900-01-01 01:14:12.605986  1900-01-01 01:14:14.605986   
1658  1900-01-01 01:14:16.605986  1900-01-01 01:14:18.605986   
1659  1900-01-01 01:14:18.605986  1900-01-01 01:14:20.605986   

                      Sentence                                Translation  \
0     مثلاً خیلی ما دوست داریم            For example, we really like it.   
1        خیلی هم دوستشون داریم                    We really love them too   


### add sentiment column

In [34]:
sentiment_path = "/Users/apple/Desktop/Transcribes/sentiment_results.csv"

sentiment_df = pd.read_csv(sentiment_path)

df["Sentiment"] = sentiment_df["Sentiment"]

print(df.head(10))


                   Start Time                    End Time  \
0  1900-01-01 00:09:18.657982  1900-01-01 00:09:20.657982   
1  1900-01-01 00:09:20.657982  1900-01-01 00:09:22.657982   
2  1900-01-01 00:09:22.657982  1900-01-01 00:09:24.657982   
3  1900-01-01 00:09:24.657982  1900-01-01 00:09:26.657982   
4  1900-01-01 00:09:26.657982  1900-01-01 00:09:28.657982   
5  1900-01-01 00:09:28.657982  1900-01-01 00:09:30.657982   
6  1900-01-01 00:09:30.657982  1900-01-01 00:09:32.657982   
7  1900-01-01 00:09:32.657982  1900-01-01 00:09:34.657982   
8  1900-01-01 00:09:34.657982  1900-01-01 00:09:36.657982   
9  1900-01-01 00:09:36.657982  1900-01-01 00:09:38.657982   

                      Sentence                                Translation  \
0     مثلاً خیلی ما دوست داریم            For example, we really like it.   
1        خیلی هم دوستشون داریم                    We really love them too   
2       خیلی هم کنار همدیگه هم     Very much together with each other too   
3      خیلی به همون 

### Sentence Length

In [35]:
df["Sentence_Length"] = df["Sentence"].astype(str).apply(lambda x: len(x.split()))

print(df[["Sentence", "Sentence_Length"]].head(10))


                      Sentence  Sentence_Length
0     مثلاً خیلی ما دوست داریم                5
1        خیلی هم دوستشون داریم                4
2       خیلی هم کنار همدیگه هم                5
3      خیلی به همون خوش میگذره                5
4             حالا چطور شغل من                4
5         چطور فضای اجتماعی من                4
6         منظور عرض من این بود                5
7         که اون آدمهایی رو که                5
8      میتونیم روشون حساب کنیم                4
9  شاید اون دوست هم کسایی باشن                6


### Presence of negation

In [40]:
negations = [
    "نه", "نیست", "نمی", "نکن", "هرگز", "ندان", "نخواه", "نشد", "بدون",
    "هیچ", "هیچ‌وقت", "اصلاً", "ابداً", "نمی‌دانم", "ندانم", "نمیدونم", 
    "نمی‌شود", "نمی‌شه", "نمیشه", "نخواهم", "نخواهیم", "نمی‌کنم", "نکردم", 
    "نخواهد", "نخواهند", "نمیتونم", "نمی‌توانم", "نمی‌خواهم", "ناممکن", 
    "محال", "غیرممکن", "منفی", "اجتناب", "اجتنابی", "ممنوع", "باطل", 
    "رد", "لغو"
]

def check_negation(sentence):
    sentence = str(sentence)
    for neg in negations:
        if neg in sentence:
            return 1   
    return 0          

df["Has_Negation"] = df["Sentence"].apply(check_negation)

print(df[["Sentence", "Has_Negation"]].head(10))


                      Sentence  Has_Negation
0     مثلاً خیلی ما دوست داریم             0
1        خیلی هم دوستشون داریم             0
2       خیلی هم کنار همدیگه هم             0
3      خیلی به همون خوش میگذره             0
4             حالا چطور شغل من             0
5         چطور فضای اجتماعی من             0
6         منظور عرض من این بود             0
7         که اون آدمهایی رو که             0
8      میتونیم روشون حساب کنیم             0
9  شاید اون دوست هم کسایی باشن             0


In [42]:
print([df])

[                      Start Time                    End Time  \
0     1900-01-01 00:09:18.657982  1900-01-01 00:09:20.657982   
1     1900-01-01 00:09:20.657982  1900-01-01 00:09:22.657982   
2     1900-01-01 00:09:22.657982  1900-01-01 00:09:24.657982   
3     1900-01-01 00:09:24.657982  1900-01-01 00:09:26.657982   
4     1900-01-01 00:09:26.657982  1900-01-01 00:09:28.657982   
...                          ...                         ...   
1655  1900-01-01 01:14:08.605986  1900-01-01 01:14:10.605986   
1656  1900-01-01 01:14:10.605986  1900-01-01 01:14:12.605986   
1657  1900-01-01 01:14:12.605986  1900-01-01 01:14:14.605986   
1658  1900-01-01 01:14:16.605986  1900-01-01 01:14:18.605986   
1659  1900-01-01 01:14:18.605986  1900-01-01 01:14:20.605986   

                      Sentence                                Translation  \
0     مثلاً خیلی ما دوست داریم            For example, we really like it.   
1        خیلی هم دوستشون داریم                    We really love them too   

In [43]:
df_10 = df.head(10).copy()

# TSV
output_path = "/Users/apple/Desktop/Transcribes/NLP_features.tsv"
df_10.to_csv(output_path, sep="\t", index=False)

print(f"File saved at: {output_path}")


File saved at: /Users/apple/Desktop/Transcribes/NLP_features.tsv


In [45]:
import csv

df_10 = df.head(10).copy()

# CSV
output_path = "/Users/apple/Desktop/Transcribes/NLP_features.csv"
df_10.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)

print(f"File saved at: {output_path}")


File saved at: /Users/apple/Desktop/Transcribes/NLP_features.csv
